In [5]:
import pandas as pd

# Load your dataset
df = pd.read_csv("Motor_Vehicle_Collisions_-_Crashes_20260430.csv")

print(df.shape)
print(df.head())

C:\Users\yossef\AppData\Local\Temp\ipykernel_17580\196302239.py:4: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Motor_Vehicle_Collisions_-_Crashes_20260430.csv")


(2258315, 29)
   CRASH DATE CRASH TIME   BOROUGH ZIP CODE  LATITUDE  LONGITUDE  \
0  09/11/2021       2:39       NaN      NaN       NaN        NaN   
1  03/26/2022      11:45       NaN      NaN       NaN        NaN   
2  11/01/2023       1:29  BROOKLYN  11230.0  40.62179 -73.970024   
3  06/29/2022       6:55       NaN      NaN       NaN        NaN   
4  09/21/2022      13:21       NaN      NaN       NaN        NaN   

                     LOCATION           ON STREET NAME CROSS STREET NAME  \
0                         NaN    WHITESTONE EXPRESSWAY         20 AVENUE   
1                         NaN  QUEENSBORO BRIDGE UPPER               NaN   
2      (40.62179, -73.970024)            OCEAN PARKWAY          AVENUE K   
3                         NaN       THROGS NECK BRIDGE               NaN   
4                         NaN          BROOKLYN BRIDGE               NaN   

  OFF STREET NAME  ...  CONTRIBUTING FACTOR VEHICLE 2  \
0             NaN  ...                    Unspecified   
1     

In [9]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print(df.columns)

Index(['crash_date', 'crash_time', 'borough', 'zip_code', 'latitude',
       'longitude', 'location', 'on_street_name', 'cross_street_name',
       'off_street_name', 'number_of_persons_injured',
       'number_of_persons_killed', 'number_of_pedestrians_injured',
       'number_of_pedestrians_killed', 'number_of_cyclist_injured',
       'number_of_cyclist_killed', 'number_of_motorist_injured',
       'number_of_motorist_killed', 'contributing_factor_vehicle_1',
       'contributing_factor_vehicle_2', 'contributing_factor_vehicle_3',
       'contributing_factor_vehicle_4', 'contributing_factor_vehicle_5',
       'collision_id', 'vehicle_type_code_1', 'vehicle_type_code_2',
       'vehicle_type_code_3', 'vehicle_type_code_4', 'vehicle_type_code_5'],
      dtype='object')


In [10]:
# Convert date
df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")

# Remove rows with bad dates
df = df.dropna(subset=["crash_date"])

# Fill missing text values
text_cols = [
    "borough",
    "zip_code",
    "on_street_name",
    "cross_street_name",
    "off_street_name",
    "contributing_factor_vehicle_1",
    "vehicle_type_code_1"
]

for col in text_cols:
    df[col] = df[col].fillna("Unknown")

# Fill missing injury/death values with 0
num_cols = [
    "number_of_persons_injured",
    "number_of_persons_killed",
    "number_of_pedestrians_injured",
    "number_of_pedestrians_killed",
    "number_of_cyclist_injured",
    "number_of_cyclist_killed",
    "number_of_motorist_injured",
    "number_of_motorist_killed"
]

df[num_cols] = df[num_cols].fillna(0)

# Remove duplicate rows
df = df.drop_duplicates()

print(df.shape)
print(df.head())

(2258315, 29)
  crash_date crash_time   borough zip_code  latitude  longitude  \
0 2021-09-11       2:39   Unknown  Unknown       NaN        NaN   
1 2022-03-26      11:45   Unknown  Unknown       NaN        NaN   
2 2023-11-01       1:29  BROOKLYN  11230.0  40.62179 -73.970024   
3 2022-06-29       6:55   Unknown  Unknown       NaN        NaN   
4 2022-09-21      13:21   Unknown  Unknown       NaN        NaN   

                     location           on_street_name cross_street_name  \
0                         NaN    WHITESTONE EXPRESSWAY         20 AVENUE   
1                         NaN  QUEENSBORO BRIDGE UPPER           Unknown   
2      (40.62179, -73.970024)            OCEAN PARKWAY          AVENUE K   
3                         NaN       THROGS NECK BRIDGE           Unknown   
4                         NaN          BROOKLYN BRIDGE           Unknown   

  off_street_name  ...  contributing_factor_vehicle_2  \
0         Unknown  ...                    Unspecified   
1         Un

In [11]:
# -----------------------------
# DIM_DATE
# -----------------------------
dim_date = df[["crash_date"]].drop_duplicates().copy()

dim_date["date_key"] = dim_date["crash_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["full_date"] = dim_date["crash_date"].dt.date
dim_date["year"] = dim_date["crash_date"].dt.year
dim_date["quarter"] = dim_date["crash_date"].dt.quarter
dim_date["month"] = dim_date["crash_date"].dt.month
dim_date["day"] = dim_date["crash_date"].dt.day
dim_date["day_of_week"] = dim_date["crash_date"].dt.day_name()

dim_date = dim_date[[
    "date_key", "full_date", "year", "quarter", "month", "day", "day_of_week"
]]

print(dim_date.head())
print(dim_date.shape)

   date_key   full_date  year  quarter  month  day day_of_week
0  20210911  2021-09-11  2021        3      9   11    Saturday
1  20220326  2022-03-26  2022        1      3   26    Saturday
2  20231101  2023-11-01  2023        4     11    1   Wednesday
3  20220629  2022-06-29  2022        2      6   29   Wednesday
4  20220921  2022-09-21  2022        3      9   21   Wednesday
(5048, 7)


In [12]:
# -----------------------------
# DIM_LOCATION
# -----------------------------
location_cols = [
    "borough",
    "zip_code",
    "latitude",
    "longitude",
    "on_street_name",
    "cross_street_name",
    "off_street_name"
]

dim_location = df[location_cols].drop_duplicates().reset_index(drop=True)
dim_location["location_key"] = dim_location.index + 1

dim_location = dim_location[[
    "location_key",
    "borough",
    "zip_code",
    "latitude",
    "longitude",
    "on_street_name",
    "cross_street_name",
    "off_street_name"
]]

print(dim_location.head())
print(dim_location.shape)

   location_key   borough zip_code  latitude  longitude  \
0             1   Unknown  Unknown       NaN        NaN   
1             2   Unknown  Unknown       NaN        NaN   
2             3  BROOKLYN  11230.0  40.62179 -73.970024   
3             4   Unknown  Unknown       NaN        NaN   
4             5   Unknown  Unknown       NaN        NaN   

            on_street_name cross_street_name off_street_name  
0    WHITESTONE EXPRESSWAY         20 AVENUE         Unknown  
1  QUEENSBORO BRIDGE UPPER           Unknown         Unknown  
2            OCEAN PARKWAY          AVENUE K         Unknown  
3       THROGS NECK BRIDGE           Unknown         Unknown  
4          BROOKLYN BRIDGE           Unknown         Unknown  
(683483, 8)


In [13]:
# -----------------------------
# DIM_VEHICLE
# -----------------------------
dim_vehicle = df[["vehicle_type_code_1"]].drop_duplicates().reset_index(drop=True)
dim_vehicle["vehicle_key"] = dim_vehicle.index + 1

dim_vehicle = dim_vehicle[[
    "vehicle_key",
    "vehicle_type_code_1"
]]

print(dim_vehicle.head())
print(dim_vehicle.shape)

   vehicle_key                  vehicle_type_code_1
0            1                                Sedan
1            2                                Moped
2            3  Station Wagon/Sport Utility Vehicle
3            4                              Unknown
4            5                                 Dump
(1881, 2)


In [14]:
# -----------------------------
# DIM_CONTRIBUTING_FACTOR
# -----------------------------
dim_factor = df[["contributing_factor_vehicle_1"]].drop_duplicates().reset_index(drop=True)
dim_factor["factor_key"] = dim_factor.index + 1

dim_factor = dim_factor[[
    "factor_key",
    "contributing_factor_vehicle_1"
]]

print(dim_factor.head())
print(dim_factor.shape)

   factor_key contributing_factor_vehicle_1
0           1  Aggressive Driving/Road Rage
1           2             Pavement Slippery
2           3                   Unspecified
3           4         Following Too Closely
4           5           Passing Too Closely
(62, 2)


In [16]:
fact = df.copy()

# Add date_key
fact["date_key"] = fact["crash_date"].dt.strftime("%Y%m%d").astype(int)

# Add location_key
fact = fact.merge(
    dim_location,
    on=[
        "borough",
        "zip_code",
        "latitude",
        "longitude",
        "on_street_name",
        "cross_street_name",
        "off_street_name"
    ],
    how="left"
)

# Add vehicle_key
fact = fact.merge(
    dim_vehicle,
    on="vehicle_type_code_1",
    how="left"
)

# Add factor_key
fact = fact.merge(
    dim_factor,
    on="contributing_factor_vehicle_1",
    how="left"
)

# Create surrogate key for fact table
fact["crash_fact_id"] = range(1, len(fact) + 1)

# Final fact table
fact_crash = fact[[
    "crash_fact_id",
    "date_key",
    "location_key",
    "vehicle_key",
    "factor_key",
    "number_of_persons_injured",
    "number_of_persons_killed",
    "number_of_pedestrians_injured",
    "number_of_pedestrians_killed",
    "number_of_cyclist_injured",
    "number_of_cyclist_killed",
    "number_of_motorist_injured",
    "number_of_motorist_killed"
]]

print(fact_crash.head())
print(fact_crash.shape)

   crash_fact_id  date_key  location_key  vehicle_key  factor_key  \
0              1  20210911             1            1           1   
1              2  20220326             2            1           2   
2              3  20231101             3            2           3   
3              4  20220629             4            1           4   
4              5  20220921             5            3           5   

   number_of_persons_injured  number_of_persons_killed  \
0                        2.0                       0.0   
1                        1.0                       0.0   
2                        1.0                       0.0   
3                        0.0                       0.0   
4                        0.0                       0.0   

   number_of_pedestrians_injured  number_of_pedestrians_killed  \
0                              0                             0   
1                              0                             0   
2                              0      

In [17]:
# Create output folder if it doesn't exist
import os
os.makedirs("HW2/output", exist_ok=True)

# Export dimension tables
dim_date.to_csv("HW2/output/dim_date.csv", index=False)
dim_location.to_csv("HW2/output/dim_location.csv", index=False)
dim_vehicle.to_csv("HW2/output/dim_vehicle.csv", index=False)
dim_factor.to_csv("HW2/output/dim_contributing_factor.csv", index=False)

# Export fact table
fact_crash.to_csv("HW2/output/fact_crash.csv", index=False)

print("All files exported successfully")

All files exported successfully
